In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
from mplex import Grid
from utils import c2xy, xy2c, get_heading

In [ ]:
data_dir = Path("data")
output_dir = Path("outputs/4_metrics_ball_displacements")
output_dir.mkdir(exist_ok=True, parents=True)
fps = 29
t = np.arange(-60, 60) / fps
n_frames = len(t)
data_paths = sorted(Path("cached").glob("*contacts.feather"))

In [ ]:
end_frames = {}
for data_path in data_paths:
    for fly, df_ in pd.read_feather(data_path)["b", "y"].groupby("fly"):
        end_frames[fly] = df_.index.get_level_values(1)[::120].values[(df_.values.reshape(-1, 120).max(1) < 100).argmax()]

In [ ]:
def get_df():
    dfs: list[pd.DataFrame] = []
    for path in tqdm(data_paths):
        df = pd.read_feather(path)
        y_ball = df[("b", "y")].values
        y_ball_mid = y_ball[n_frames // 2::n_frames]
        y_ball_end = y_ball[n_frames - 1::n_frames]
        diff_y_ball = np.abs(y_ball_end - y_ball_mid)
        indices = (np.where(diff_y_ball > 8)[0][:, None] * 120 + np.arange(n_frames)).ravel()
        df = xy2c(df.iloc[indices]) * 1j

        heading = get_heading(c2xy(df[["h", "t", "a"]].values))
        front_leg = ((df["lf"].values + df["rf"].values) / 2 - df["t"].values) / heading
        df = pd.DataFrame(
            {
                "ball": df["b"].values.real,
                "ball_fly_distance": df["b"].values.real - df["t"].values.real,
                "front_leg_x": front_leg.real,
                "heading": np.angle(heading, deg=True),
            },
            index=df.index,
        )
        n_features = df.shape[1]
        df = pd.DataFrame(
            df.values.reshape((-1, n_frames, n_features))
            .transpose((0, 2, 1))
            .reshape((-1, n_features * n_frames)),
            index=df.index[::n_frames],
            columns=pd.MultiIndex.from_product(
                [df.columns, np.arange(n_frames)], names=["feature", "frame"]
            ),
        )
        for col in ["ball",]:
            df[col] -= df[col].iloc[:, n_frames // 2].values[:, None]

        dfs.append(df)

    return pd.concat(dfs)

df = get_df()
df_flies = pd.concat(
    [pd.read_feather(path) for path in sorted(Path("cached").glob("*flies.feather"))]
).set_index("fly")

In [ ]:
df_split = pd.read_csv(data_dir / "split.csv", index_col="line")
df_genotypes = pd.read_csv(data_dir / "genotypes.csv", index_col="genotype")
df_genotypes["split"] = df_split.loc[df_genotypes.index, "split"]
controls = {"y": "Empty split-GAL4", "n": "Empty GAL4", "m": "TNT×PR"}
event_display_names = df_genotypes.loc[df_flies.loc[df.index.get_level_values(0), "line"], "display_name"].values

In [ ]:
unused_lines = df_genotypes[df_genotypes["display_name"].isin(["TH", "PR", "Canton-S"])].index
df = df[~df_flies.loc[df.index.get_level_values(0), "line"].isin(unused_lines).values]
df_flies = df_flies[~df_flies["line"].isin(unused_lines)]

In [ ]:
is_before_end = np.zeros((len(df),), dtype=bool)
for fly, frame in tqdm(list(end_frames.items())):
    is_before_end[(df.index.get_level_values(0) == fly) & (df.index.get_level_values(1) <= frame)] = True

In [ ]:
import numba
from pynndescent.distances import euclidean


@numba.njit
def flip(x):
    x = x.copy()
    x[-120:] *= -1
    return x


@numba.njit
def my_dist(a, b):
    return min(euclidean(a, b), euclidean(a, flip(b)))

X = df.values.copy()
X = X / np.repeat(X.reshape((len(X), -1, n_frames)).std((0, 2)), n_frames)

if Path(output_dir / "Z.npy").exists():
    Z = np.load(output_dir / "Z.npy")
else:
    from umap import UMAP

    umap = UMAP(n_components=2, random_state=0, verbose=True, metric=my_dist)
    Z = umap.fit_transform(X)
    np.save(output_dir / "Z.npy", Z)

In [ ]:
from utils import get_kde, get_non_outliers

non_outlier, mask, bound = get_non_outliers(Z)
g = Grid(80)
ax = g.item()
ax.imshow(
    mask,
    cmap="gray_r",
    alpha=0.3,
    extent=(-bound, bound, -bound, bound),
    origin="lower",
)
ax.scatter(*Z.T, s=0.5, alpha=0.1, marker=".", lw=0, c=non_outlier)

In [ ]:
from utils import rotate_embedding

X = X[non_outlier]
Z = Z[non_outlier]
df = df[non_outlier]
Z = rotate_embedding(Z)
bound = np.abs(Z).max() * 1.05
im = get_kde(Z, bound=bound)[0]
n_bins = len(im)

In [ ]:
from utils import cluster_points, sort_labels

n_clusters = 20
labels = cluster_points(Z, n_clusters=n_clusters)
labels = (labels - 1) % n_clusters

In [ ]:
labels = sort_labels(labels, Z)

In [ ]:
from utils import get_areas

density_threshold = 5e-5
im_regions, xlim, ylim = get_areas(Z, labels, bound, n_bins, density_threshold)

In [ ]:
xlim = xlim[::-1]

In [ ]:
%matplotlib inline

In [ ]:
from utils import plot_map_regions

g = plot_map_regions(im_regions, Z, labels, xlim, ylim, bound)
bar_length = 2.5
ax = g.item()
ax.add_scale_bars(
    xlim[1] + 1,
    ylim[0] + 0.3,
    -bar_length if xlim[1] > xlim[0] else bar_length,
    bar_length if ylim[1] > ylim[0] else -bar_length,
    xlabel="UMAP 1",
    ylabel="UMAP 2",
    fmt="",
    pad=(1.5, -4.5),
    size=4,
    text_kw=dict(color="k"),
    lw=0.3,
    c="k",
)
g.savefig(output_dir / "regions.pdf")

In [ ]:
from utils import get_need_flip

need_flip = get_need_flip(X, Z, labels, flip)

In [ ]:
df = pd.DataFrame(
    np.array([flip(xi) if need_flip[i] else xi for i, xi in enumerate(df.values)]),
    columns=df.columns,
    index=df.index,
)

In [ ]:
for k in range(n_clusters):
    mask = labels == k
    mean_heading = df.loc[mask, "heading"].values.mean()
    if mean_heading < 0:
        df.loc[mask] = np.array([flip(xi) for xi in df.values[mask]])
        need_flip[mask] = ~need_flip[mask]

In [ ]:
features = list(df.columns[::n_frames].get_level_values(0))
n_features = len(features)
feature_names = [
    "Ball horizontal\ndisplacement\n(px)",
    "Ball-fly\nhorizontal\ndistance (px)",
    "Front leg\nhorizontal\nposition (px)",
    "Heading/\ninclination\nangle (°)",
]

In [ ]:
from matplotlib.ticker import MaxNLocator
from utils import get_cluster_palette

cluster_palette = get_cluster_palette(n_clusters)
cluster_ids = np.arange(n_clusters)
g = Grid(
    (8, 12),
    (n_features, len(cluster_ids)),
    sharex=True,
    sharey="row",
    spines="lb",
    space=(2, 2),
)
g[:, 1:].set_visible_sides("")
g[:, 0].set_visible_sides("l")

for i_cluster, cluster_id in enumerate(cluster_ids):
    ax = g[0, i_cluster]
    ax.set_title(cluster_id + 1, size=4, pad=0)
    for i, (feature, title) in enumerate(zip(features, feature_names)):
        mu = df[feature].values[labels == cluster_id].mean(0)
        s = df[feature].values[labels == cluster_id].std(0)
        ax = g[i, i_cluster]
        c = cluster_palette[cluster_id]
        ax.plot(t, mu, color=c, lw=1)
        ax.fill_between(t, mu - s, mu + s, alpha=0.5, color=c, lw=0)
        if i_cluster == 0:
            ax.add_text(
                0,
                0.5,
                title,
                ha="r",
                va="c",
                transform="a",
                pad=(-8, 0),
                size=4,
            )
        ax.tick_params("y", labelsize=3)
        ax.yaxis.set_major_locator(MaxNLocator(3, integer=True))

ax = g[-1, 0]
ax.set_xlabel("Time (s)", size=4)
ax.tick_params("x", labelsize=3)
g.set_xlim(t[0], t[-1])
ax.set_xticks([-2, 0, 2])
ax.set_visible_sides("lb")
g[0].make_ax().add_text(0.5, 1, "Cluster (#)", ha="c", va="b", pad=(0, 6), size=4)
g.savefig(output_dir / "features.pdf")
ylims = np.array([ax.get_ylim() for ax in g.axs[:, 0]])

In [ ]:
from tqdm import tqdm

In [ ]:
video_dir = Path("event_videos")

In [ ]:
%matplotlib inline

In [ ]:
def get_plot(k, dpi):
    from matplotlib.colors import to_hex

    g = Grid(
        ([14, 20], [40] + [20] * n_features),
        facecolor="k",
        sharex=False,
        sharey=False,
        spines="l",
        space=(10, [2] + [2] * (n_features - 1)),
        border=(2, 2, 2, 15),
        dpi=dpi,
    )
    axs = g.axs[1:, 1]
    txt_kw = dict(color="w", transform="a", size=3, va="c")
    for i, (feature, title) in enumerate(zip(features, feature_names)):
        ax = axs[i]
        mu = df[feature].values[labels == k].mean(0)
        s = df[feature].values[labels == k].std(0)
        c = "w"
        ax.plot(t, mu, color=c, lw=1)
        ax.fill_between(t, mu - s, mu + s, alpha=0.5, color=c, lw=0)
        ax.add_text(0, 0.5, title, pad=(-7, 0), rotation=0, ha="r", **txt_kw)
        ax.yaxis.set_major_locator(MaxNLocator(3, integer=True))
        ax.tick_params(axis="both", colors="w", labelsize=2.5)

        if i == n_features - 1:
            ax.set_xlabel("Time (s)", size=3, color="w")
            g.set_xlim(t[0], t[-1])
            ax.set_xticks([-2, 0, 2])
        else:
            ax.set_xticks([])

        for spine in ax.spines.values():
            spine.set_color("w")

    g[-1].set_visible_sides("lb")
    g[0].set_visible_sides("")

    ax = g[0].make_ax(behind=False)
    ax.add_text(0.5, 1, f"Cluster {k + 1}", pad=(0, 2), ha="c", **dict(txt_kw, size=4))
    ax.scatter(*Z.T, s=1, c=cluster_palette[labels], alpha=0.05, lw=0, marker=".")
    cnt_kws = dict(antialiased=True, origin="lower", extent=(-bound, bound) * 2)
    ax.contour(im_regions + 1, levels=np.arange(n_clusters + 1), colors=to_hex((0.5,) * 3), **cnt_kws)
    ax.contour(im_regions == k, colors="w", linewidths=0.5, zorder=100, **cnt_kws)
    bar_length = 3
    ax.add_scale_bars(
        xlim[1] + 0.5,
        ylim[0] - 0,
        -bar_length if xlim[1] > xlim[0] else bar_length,
        bar_length if ylim[1] > ylim[0] else -bar_length,
        xlabel="UMAP 1",
        ylabel="UMAP 2",
        fmt="",
        pad=(1.5, -2.5),
        size=2,
        text_kw=dict(color="w"),
        lw=0.3,
        c="w",
    )
    ax.set_aspect("equal")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    axt = g[1:, 1].make_ax(sharex=axs[0], behind=True)
    t_line = axt.axvline(t[0], color="w", lw=0.25, ls="--", animated=True)
    return g, t_line, axt

dpi = 508.6092715231788
g, t_line, axt = get_plot(0, dpi=dpi)
im0 = g.to_rgba_array()[..., :3]
plt.close(g.fig)
plt.imshow(im0)
plt.gca().axis("off")

In [ ]:
from utils import make_video

def get_df_orig():
    dfs = []
    for path in tqdm(data_paths):
        df = pd.read_feather(path)
        y_ball = df[("b", "y")].values
        y_ball_mid = y_ball[n_frames // 2::n_frames]
        y_ball_end = y_ball[n_frames - 1::n_frames]
        diff_y_ball = np.abs(y_ball_end - y_ball_mid)
        indices = (np.where(diff_y_ball > 8)[0][:, None] * n_frames + np.arange(n_frames)).ravel()
        df = df.iloc[indices]
        dfs.append(df)
    return pd.concat(dfs)


df_orig = get_df_orig()

In [ ]:
from video_reader import PyVideoReader
from utils import choice, edges, fly_palette
from matplotlib.transforms import Affine2D

ball_color = "#00aeef"

selected_clips = choice(np.where(labels == 11)[0], 16 * 6, seed=0)
fly, event_id, frame_id = df.index[selected_clips[6 * 6]]
fly_dir = Path(df_flies.loc[fly, "path"])
video_path = (fly_dir / f"{fly_dir.stem}_preprocessed.mp4").as_posix()
offset = 38
im = PyVideoReader(video_path)[frame_id + offset]

ball_rel_pos_x = 0.7
ball_rel_pos_y = 0.5
width = 256
height = 64
zoom = 1.5
thickness = 1
ball_radius = 12
df_ = df_orig.loc[fly, event_id, frame_id : frame_id + n_frames]
cx, cy = df_["b"].iloc[offset].astype(int)
ball_pos_x = ball_rel_pos_x * width
ball_pos_y = ball_rel_pos_y * height

g = Grid()
ax = g.item()
trans = Affine2D().translate(-cx, -cy).rotate_deg(90) + ax.transData
ax.imshow(im, transform=trans)
xy = df_.iloc[offset].unstack()
for src, dst in edges[:-2]:
    pt1 = tuple(map(round, (*xy.loc[src], 1)))
    pt2 = tuple(map(round, (*xy.loc[dst], 1)))
    ax.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]], color=fly_palette[dst], transform=trans, lw=thickness, solid_capstyle="round")
ax.set_xlim(-80, 40)
ax.set_ylim(16, -16)
xy = tuple(map(round, (*xy.loc["b"], 1)))
from matplotlib.patches import Circle
ax.add_patch(Circle(xy, ball_radius, edgecolor=ball_color, facecolor="none", lw=thickness, transform=trans))
ax.axis("off")
g.savefig(output_dir / f"{fly}_{event_id}_{frame_id}.pdf")

In [ ]:
kwargs = dict(
    labels=labels,
    n_frames=n_frames,
    df=df,
    df_flies=df_flies,
    df_orig=df_orig,
    need_flip=need_flip,
    t=t,
    get_plot=get_plot,
    dpi=dpi,
    n_rows=8,
    n_cols=3,
    zoom=3,
    thickness=2,
    width=512,
    height=128,
    ball_radius=32,
    save_dir=output_dir / "videos_8x3",
)

In [ ]:
from utils import concat_videos
from joblib import Parallel, delayed
from tqdm import trange

seeds = np.zeros(n_clusters, dtype=int)
Parallel(n_jobs=-1)(delayed(make_video)(k=k, seed=seeds[k], **kwargs) for k in trange(n_clusters))
concat_videos(sorted(kwargs["save_dir"].glob("cluster_*.mp4")), output_dir / f'{kwargs["save_dir"].stem}.mp4')

In [ ]:
seeds = np.zeros(n_clusters, dtype=int)
kwargs.update(
    dict(
        n_rows=16,
        n_cols=6,
        zoom=1.5,
        thickness=1,
        width=256,
        height=64,
        ball_radius=16,
        save_dir=output_dir / "videos_16x6",
        ball_color="#00aeef",
    )
)

seeds = np.zeros(n_clusters, dtype=int)
Parallel(n_jobs=-1)(
    delayed(make_video)(k=k, seed=seeds[k], **kwargs) for k in trange(n_clusters)
)
concat_videos(sorted(kwargs["save_dir"].glob("cluster_*.mp4")), output_dir / f'{kwargs["save_dir"].stem}.mp4')

In [ ]:
%matplotlib inline

In [ ]:
from natsort import natsorted

regions = [
    "Control",
    "Vision",
    "Olfaction",
    "fchON",
    "JON",
    "LH",
    "CX",
    "MB",
    "MB extrinsic neurons",
    "Neuropeptide",
    "DN",
]
controls = {"y": "Empty-Split-Gal4", "n": "Empty-Gal4", "m": "TNTxPR"}
df_genotypes = pd.read_csv("data/BallPushing_TNTScreen - Simplified_info.csv")
event_lines = (
    df_genotypes.set_index("Genotype")
    .loc[
        df_flies.loc[df.index.get_level_values(0), "line"].values, "Simplified Nickname"
    ]
    .values
)
df_lines = df_genotypes.iloc[:, [-2, -3, -1]].drop_duplicates()
df_lines.columns = ["line", "region", "split"]
df_lines = df_lines[df_lines["line"].isin(set(event_lines))]
df_lines = pd.concat(
    [
        df_lines[df_lines["split"].eq(split) & df_lines["region"].eq(region)]
        .set_index("line")
        .loc[
            natsorted(
                df_lines[df_lines["split"].eq(split) & df_lines["region"].eq(region)][
                    "line"
                ]
            )
        ]
        for split in controls
        for region in regions
    ]
)

In [ ]:
event_times = df.index.get_level_values(2).values / fps

In [ ]:
ZT = np.column_stack((Z, event_times))
ZT[:, :2] /= ZT[:, 2].std()
ZT[:, -1] /= ZT[:, -1].std()

def energy_distance(X, Y):
    from scipy.spatial.distance import cdist
    XY = cdist(X, Y)
    XX = cdist(X, X)
    YY = cdist(Y, Y)
    return 2 * XY.mean() - XX.mean() - YY.mean()

d = np.zeros((len(df_lines), len(df_lines)))
args = []

for i, line_i in enumerate(df_lines.index):
    for j, line_j in enumerate(df_lines.index):
        if i < j:
            args.append((i, j, line_i, line_j))

for arg in tqdm(args):
    i, j, line_i, line_j = arg
    dist = energy_distance(ZT[(event_lines == line_i) & is_before_end], ZT[(event_lines == line_j) & is_before_end])
    d[i, j] = dist
    d[j, i] = dist

import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform

d = squareform(d)
Z_linkage = sch.linkage(d, method="average", optimal_ordering=True)
leaf_order = sch.leaves_list(Z_linkage)

In [ ]:
maxclust = 12
fclust = sch.fcluster(Z_linkage, t=maxclust, criterion="maxclust") - 1
fclust.min()
fclust_ = fclust.copy()
for i, j in enumerate(np.argsort([np.arange(len(leaf_order))[np.isin(leaf_order, np.arange(len(leaf_order))[fclust == i])].mean() for i in range(maxclust)])):
    fclust_[fclust == j] = i
fclust = fclust_
np.argsort([np.arange(len(leaf_order))[np.isin(leaf_order, np.arange(len(leaf_order))[fclust == i])].mean() for i in range(maxclust)])

In [ ]:
from utils import energy_test_fly

def func(display_name):
    control = controls[df_lines.loc[display_name, "split"]]
    results = energy_test_fly(
        Z[event_lines == control],
        Z[event_lines == display_name],
        df.index.get_level_values(0)[event_lines == control],
        df.index.get_level_values(0)[event_lines == display_name],
        n_samples=10000,
        random_state=0,
    )
    return (display_name, float(results[1]), float(results[0]))

from joblib import Parallel, delayed

pval_stats = Parallel(n_jobs=-1)(delayed(func)(display_name) for display_name in tqdm(df_lines.index))

In [ ]:
df_test = pd.DataFrame(pval_stats, columns=["line", "pval", "stat"]).set_index("line")
df_test

In [ ]:
df_test.to_csv(output_dir / "energy_test_results.csv")

In [ ]:
df_test.loc["LH272"]

In [ ]:
from statsmodels.stats.multitest import multipletests

_, df_test["pval_corr"], _, _ = multipletests(df_test["pval"].values, alpha=0.05, method='fdr_bh')

In [ ]:
n_flies = df_genotypes.set_index("Genotype").loc[df_flies["line"], "Simplified Nickname"].value_counts()
n_events = pd.DataFrame({"fly": df.index.get_level_values(0), "line" : df_genotypes.set_index("Genotype").loc[df_flies.loc[df.index.get_level_values(0), "line"], "Simplified Nickname"].values}).groupby(["line", "fly"]).size()
n_events_mean = n_events.groupby("line").mean()
n_events_std = n_events.groupby("line").std()

In [ ]:
from utils import split_list, region_palette
from matplotlib.colors import to_hex

for split_type in ["y", "n", "m"]:
    line_list = list(df_lines[df_lines["split"].eq(split_type)].index)
    line_list.remove(controls[split_type])
    line_list = [controls[split_type]] + line_list
    line_list = df_test.loc[line_list].sort_values(["pval_corr", "stat"], ascending=[False, True]).index.tolist()
    h = (np.abs(np.diff(ylim) / np.diff(xlim)) * 60).item()
    n_cols = 10
    n_rows = int(np.ceil(len(line_list) / n_cols))
    if n_rows == 1:
        n_cols = len(line_list)
    cmap = "gray_r"
    g = Grid((60, h), (n_rows, n_cols), space=(6, 24), facecolor="w")
    axs = g.axs.ravel()
    for ax, line in zip(axs, line_list):
        bidx = event_lines == line
        # x, y = Z[bidx].T
        im_kde = get_kde(Z[bidx], bound=bound, bw=0.4)[0]
        im_kde /= im_kde.mean()
        ax.imshow(
            im_kde,
            cmap=cmap,
            extent=(-bound, bound, -bound, bound),
            origin="lower",
            vmin=0,
            vmax=20,
        )
        ax.contour(
            im_regions + 1,
            levels=np.arange(n_clusters + 1),
            colors=to_hex((0.3,) * 3),
            extent=(-bound, bound, -bound, bound),
            antialiased=True,
            linewidths=0.5,
            origin="lower",
        )
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_aspect("equal")

        if line == controls[split_type]:
            txt = f"{line}\n(control)"
        else:
            pval = df_test.loc[line, "pval_corr"]
            txt = f"{line}\np={pval:.3f}, E={df_test.loc[line, 'stat']:.3f}"

        txt += f"\n{n_flies.loc[line]} flies, {round(n_events_mean.loc[line])}$\\pm${round(n_events_std.loc[line])} events"
        ax.add_text(
            0.5,
            1,
            txt,
            ha="c",
            va="b",
            transform="a",
            c=region_palette[df_lines.loc[line, "region"]],
            size=6,
            pad=(0, 0),
        )

    ax = g[0, 0]
    bar_length = 3
    ax.add_scale_bars(
        xlim[1] + 0.4,
        ylim[0] + 0.5,
        -bar_length if xlim[1] > xlim[0] else bar_length,
        bar_length if ylim[1] > ylim[0] else -bar_length,
        xlabel="UMAP 1",
        ylabel="UMAP 2",
        fmt="",
        pad=(2, -4.5),
        size=3.5,
    )
    for ax in g.axs.ravel():
        ax.set_rasterized(True)

    g[0, 0].set_zorder(100)
    g.set_visible_sides("")
    g.savefig(output_dir / f"lines_{split_type}.pdf")